In [1]:
import os
import yaml
import multiprocessing as mp
import time
import tools21cm as t2c
from tqdm import tqdm
import astropy.units as u
import astropy.constants as cst
import cupy as cp
import numpy as np
import itertools

import matplotlib.pyplot as plt
plt.rcParams["font.family"] = "serif"


def filter_baselines(baselines, visibilities, u_freq, v_freq):
    """Filters baselines and corresponding visibilities based on the range of
    frequencies. This is so that the baselines outside the frequency range of
    the fft image are removed.

    Parameters:
    baselines (list): List of baseline coordinates.
    visibilities (list): List of corresponding visibilities.
    u_freq (list): List of u frequencies.
    v_freq (list): List of v frequencies.

    Returns:
    filtered_baselines (ndarray): Numpy array of filtered baseline coordinates.
    filtered_visibilities (ndarray): Numpy array of filtered visibilities.
    """
    filtered_baselines = []
    filtered_visibilities = []

    for baseline, visibility in zip(baselines, visibilities):
        x, y = baseline
        # Check if baseline coordinates are within the range of frequencies
        if (np.min(u_freq) <= x <= np.max(u_freq)) and (
            np.min(v_freq) <= y <= np.max(v_freq)
        ):
            filtered_baselines.append(baseline)
            filtered_visibilities.append(visibility)
    return np.array(filtered_baselines), np.array(filtered_visibilities)


def filter_baselines_cupy(baselines, visibilities, u_freq, v_freq):
    """Filters baselines and corresponding visibilities based on the range of
    frequencies. This is so that the baselines outside the frequency range of
    the fft image are removed.

    Parameters:
    baselines (ndarray): Numpy array of baseline coordinates.
    visibilities (ndarray): Numpy array of corresponding visibilities.
    u_freq (ndarray): Numpy array of u frequencies.
    v_freq (ndarray): Numpy array of v frequencies.

    Returns:
    filtered_baselines (ndarray): CuPy array of filtered baseline coordinates.
    filtered_visibilities (ndarray): CuPy array of filtered visibilities.
    """
    # Convert inputs to CuPy arrays
    baselines_cp = cp.array(baselines)
    visibilities_cp = cp.array(visibilities)
    u_freq_cp = cp.array(u_freq)
    v_freq_cp = cp.array(v_freq)
    
    # Find the min and max frequencies
    u_min, u_max = cp.min(u_freq_cp), cp.max(u_freq_cp)
    v_min, v_max = cp.min(v_freq_cp), cp.max(v_freq_cp)
    
    # Filter baselines based on frequency range
    x = baselines_cp[:, 0]
    y = baselines_cp[:, 1]
    mask = (u_min <= x) & (x <= u_max) & (v_min <= y) & (y <= v_max)
    
    # Apply mask to baselines and visibilities
    filtered_baselines = baselines_cp[mask]
    filtered_visibilities = visibilities_cp[mask]
    
    return filtered_baselines, filtered_visibilities


def read_config(input_path):
    with open(input_path) as stream:
        try:
            config = yaml.safe_load(stream)
        except yaml.YAMLError as exc:
            print(exc)
    return config


def plot(bare_estimator, ell_mean, var_squared, file_path, img_dpi):
    # Plot the results
    fig, axs = plt.subplots(1, 2, figsize=(17, 5))

    axs[0].errorbar(ell_mean, bare_estimator, np.sqrt(var_squared),
        color="#1f77b4",
        label="Gaussian primary beam",
        elinewidth=0.75,
        capsize=4,
        fmt="o",
        markersize=3)
    axs[0].plot(ell_mean, 5.13 * 1e-4 * (1000 / ell_mean) ** 2.34,
        color="k",
        label="Model",
        alpha=0.75)
    axs[0].set_yscale("log")
    #axs[0].set_ylim(5.13 * 1e-4 * (1000 / ell_mean[-1]) ** 2.34, 5.13 * 1e-4 * (1000 / ell_mean[0]) ** 2.34)
    axs[0].grid()
    axs[0].set_xlabel(r"$\ell$", fontsize=14)
    axs[0].set_ylabel(r"$C_{\ell}$ $[K^2]$", fontsize=14)
    axs[0].legend()

    axs[1].errorbar(ell_mean, ell_mean * (ell_mean + 1) * bare_estimator / (2 * np.pi), ell_mean * (ell_mean + 1) * np.sqrt(var_squared) / (2 * np.pi),
        color="#1f77b4",
        label="Gaussian primary beam",
        elinewidth=0.75,
        capsize=4,
        fmt="o",
        markersize=3)
    axs[1].plot(ell_mean, ell_mean * (ell_mean + 1) * 5.13 * 1e-4 * (1000 / ell_mean) ** 2.34 / (2 * np.pi),
        color="k",
        label="Model",
        alpha=0.75)
    axs[1].set_yscale("log")
    axs[1].set_xscale("log")
    # axs[1].set_ylim(ell_mean[-1] * (ell_mean[-1] + 1) * 5.13 * 1e-4 * (1000 / ell_mean[-1]) ** 2.34 / (2 * np.pi),
    #     ell_mean[0] * (ell_mean[0] + 1) * 5.13 * 1e-4 * (1000 / ell_mean[0]) ** 2.34 / (2 * np.pi))
    # axs[1].set_xlim(1e2, 1e4)
    axs[1].grid()
    axs[1].set_xlabel(r"$\ell$", fontsize=14)
    axs[1].set_ylabel(r"$\ell(\ell + 1)C_{\ell}/2\pi$ $[K^2]$", fontsize=14)
    axs[1].legend()
    # Create the folder if it does not exist
    os.makedirs(os.path.dirname(file_path), exist_ok=True)
    plt.savefig(file_path, dpi=img_dpi)
    # plt.show()


def setup(config, bls, vis, lam):
    # Determine which function to use based on the configuration
    estimator_func_choice = config.get('computation_type')
    if estimator_func_choice == 'cupy':
        bare_estimator_func = bare_estimator_func_cupy
    elif estimator_func_choice == 'noMP':
        bare_estimator_func = bare_estimator_func_v4_noMP
    elif estimator_func_choice == 'MP':
        bare_estimator_func = bare_estimator_func_v4
    else:
        print('Computation type input invalid. Can be \'MP\' for multiprocessing,',
              '\'noMP\' for no multiprocessing, and \'cupy\'.')
        
    
    binning_method = config["binning_method"]
    num_bins = config["num_bins"]
    #theta_maj = config["theta_maj"]
    #theta_min = config["theta_min"]
    #theta_fwhm= np.mean([theta_maj, theta_min])
    diameter = config["D"]
    theta_fwhm = 1.03 * lam / diameter
    
    bare_estimator, ell_mean, var_squared= bare_estimator_func(bls, vis, 
                                                theta_fwhm, num_bins, binning_method, diameter, lam)
    
    
    return bare_estimator, ell_mean, var_squared


def parse_timesteps(timesteps_config):
    if isinstance(timesteps_config, str):
        start, end = map(int, timesteps_config.split(':'))
        return list(range(start, end))
    elif isinstance(timesteps_config, list):
        return timesteps_config
    else:
        raise ValueError("Invalid format for timesteps. Use a list or a range string (e.g., '50:60').")

***
## __All functions for the cupy option.__

In [2]:
def _splice_values_equal_bins_cupy(blcoord_sorted, blmag_sorted, vis_sorted, num_bins):
    """
    The provided arrays should be cupy arrays.
    """
    # Step 1: Calculate the number of elements per bin
    num_elements = len(blmag_sorted)
    elements_per_bin = num_elements // num_bins
    extra_elements = num_elements % num_bins

    # Step 2: Create an array of bin sizes
    bin_sizes = cp.full(num_bins, elements_per_bin)
    bin_sizes[:extra_elements] += 1

    # Step 3: Calculate the cumulative sum of bin sizes
    cumulative_sizes = cp.cumsum(bin_sizes)
    cumulative_sizes_np = cp.asnumpy(cumulative_sizes)  # Convert to NumPy array

    # Step 4: Initialize the bins
    blcoord_bins = cp.split(blcoord_sorted, cumulative_sizes_np[:-1])
    blmag_bins = cp.split(blmag_sorted, cumulative_sizes_np[:-1])
    vis_bins = cp.split(vis_sorted, cumulative_sizes_np[:-1])

    return blcoord_bins, blmag_bins, vis_bins

def _splice_values_function_fit_cupy(blmag_sorted_transformed, blcoord_sorted, blmag_sorted, vis_sorted, num_bins):
    bins = cp.linspace(np.min(blmag_sorted_transformed), np.max(blmag_sorted_transformed), num_bins)
    bin_indices = np.digitize(blmag_sorted_transformed, bins) - 1
    
    # Use histogram to count the occurrences of each bin
    hist, _ = cp.histogram(bin_indices, bins=cp.arange(0, num_bins + 1))

    # Calculate the cumulative counts to determine the indices where each bin's values start and end
    cum_counts = cp.cumsum(hist)
    cumulative_sizes_np = cp.asnumpy(cum_counts)  # Convert to NumPy array

    bin_blcoords = cp.split(blcoord_sorted, cumulative_sizes_np[:-1])
    bin_vis = cp.split(vis_sorted, cumulative_sizes_np[:-1])
    bin_blmags = cp.split(blmag_sorted, cumulative_sizes_np[:-1])

    return bin_blcoords, bin_vis, bin_blmags


def compute_vals_cupy(blcoord, vis, sigma_0, V_0, bin_number):

    bin_len = len(blcoord)
    if bin_len == 0 or bin_len == 1:
        return 0, 0, 0

    # Create a meshgrid for vectorized pairwise operations
    idx_i, idx_j = np.triu_indices(bin_len, k=1)
    idx_i = cp.asarray(idx_i)
    idx_j = cp.asarray(idx_j)

    # Calculate pairwise distances
    delta= blcoord[idx_i] - blcoord[idx_j]
    distance_norm = cp.linalg.norm(delta, axis=1)
    
    combination_len_unfiltered= len(distance_norm)
    
    # Filter pairs where distance is less than or equal to sigma_0
    valid_pairs = distance_norm > sigma_0
    if not cp.any(valid_pairs):
        return 0, 0, 0

    distance_norm = distance_norm[valid_pairs]
    idx_i = idx_i[valid_pairs]
    idx_j = idx_j[valid_pairs]
    
    combination_len_filtered= len(distance_norm)
    #print("Bin", bin_number, ":Theoretical combinations: ", combination_len_unfiltered,
    #     " - When filtered: ", combination_len_filtered)
    
    # Calculate weights
    weight = cp.exp(-(distance_norm**2) / (sigma_0**2))

    # Calculate the visibility product
    visibility_product = vis[idx_i] * cp.conj(vis[idx_j])

    #Calclating the bare estimator
    be_numerator = cp.sum(weight * visibility_product)
    be_denominator = cp.sum(weight * V_0 * cp.exp(-(distance_norm**2) / (sigma_0**2)))

    #Calculating the mean of ell
    l_i = 2 * cp.pi * cp.sqrt(blcoord[idx_i, 0]**2 + blcoord[idx_i, 1]**2)
    exp_weight_distance = weight * cp.exp(-(distance_norm**2) / (sigma_0**2))

    ell_numerator = cp.sum(exp_weight_distance * l_i)
    ell_denominator = cp.sum(exp_weight_distance)

    #Computing arrays needed for the error (variance)
    Eb_square_mean_numerator = cp.sum((exp_weight_distance * 5.13 * 1e-4 * (1000 / l_i) ** 2.34) ** 2)
    Eb_mean_square_numerator = cp.sum(exp_weight_distance * 5.13 * 1e-4 * (1000 / l_i) ** 2.34)
    Eb_denominator = cp.sum(exp_weight_distance)

    bare_estimator = cp.abs(be_numerator) / be_denominator
    ell_mean = ell_numerator / ell_denominator
    var_squared = (Eb_square_mean_numerator / Eb_denominator - (Eb_mean_square_numerator / Eb_denominator) ** 2)

    # Convert results back to numpy arrays
    bare_estimator = cp.asnumpy(bare_estimator)
    ell_mean = cp.asnumpy(ell_mean)
    var_squared = cp.asnumpy(var_squared)

    return bare_estimator, ell_mean, np.abs(var_squared)


def bare_estimator_func_cupy(baselines, visibilities, theta_fwhm, num_bins, bin_type, D, lam):
    #theta_0 = 0.6 * theta_fwhm
    #V_0 = np.pi * theta_0**2 / 2
    #sigma_0 = 0.76 / theta_fwhm
    theta_fwhm = 1.03 * lam / D
    sigma_0 = 0.76 / theta_fwhm
    theta_0 = 0.6 * theta_fwhm
    V_0 = np.pi * theta_0**2 / 2


    start_time = time.time()
    # Step 0: turn arrays into cupy arrays
    baselines= cp.asarray(baselines, dtype= cp.float32)
    visibilities= cp.asarray(visibilities, dtype= cp.complex64)

    # Step 1: Calculate the baseline magnitudes
    bl_mag = cp.linalg.norm(baselines, axis=1)

    # Stack baselines, magnitudes, and visibilities into a single array for sorting
    #combined_array = cp.hstack((baselines[:,0, None], baselines[:,1, None], bl_mag[:, None], visibilities[:, None]))
    combined_array = cp.hstack((baselines[:], bl_mag[:, None], visibilities[:, None]))

    # Sort the combined array based on the magnitudes
    sorted_combined_array = combined_array[cp.argsort(combined_array[:,2])]

    # Split the sorted array back into individual components
    blcoord_sorted = sorted_combined_array[:, 0:2]
    blcoord_sorted = blcoord_sorted.astype(cp.float32)  # Change to desired dtype
    blmag_sorted = sorted_combined_array[:, 2]
    blmag_sorted = blmag_sorted.astype(cp.float32)      # Change to desired dtype
    vis_sorted = sorted_combined_array[:, -1]
    vis_sorted = vis_sorted.astype(cp.complex64)          # Change to desired dtype

    #print("Length of input arrays: ", len(blmag_sorted))

    if bin_type == "log":
        blmag_sorted_transformed = cp.log10(blmag_sorted)
        blcoord_bins, blmag_bins, vis_bins = _splice_values_function_fit_cupy(blmag_sorted_transformed, blcoord_sorted, blmag_sorted, vis_sorted, num_bins)

    elif bin_type == "linear":
        blmag_sorted_transformed = blmag_sorted
        blcoord_bins, blmag_bins, vis_bins = _splice_values_function_fit_cupy(blmag_sorted_transformed, blcoord_sorted, blmag_sorted, vis_sorted, num_bins)

    elif bin_type == "equal_length":
        blcoord_bins, blmag_bins, vis_bins = _splice_values_equal_bins_cupy(blcoord_sorted, blmag_sorted, vis_sorted, num_bins)
    else:
        raise ValueError("Unsupported bin_type")

    end_time = time.time()
    #print('Time to sort and bin values: ', end_time-start_time, 's')

    # Verify the bins
    #for i, bin in enumerate(blcoord_bins):
    #    print(f"Bin {i+1}: {len(bin)} elements - Combinations: {int(len(bin)*(len(bin)-1) / 2)}")
    # Print to verify the results
    #print("Total elements after binning: ", cp.sum(cp.array([len(bin) for bin in blmag_bins])))

    bare_estimators = np.empty(num_bins, dtype=np.float32)
    average_ells = np.empty(num_bins, dtype=np.float32)
    square_variances = np.empty(num_bins, dtype=np.float32)

    start_time = time.time()
    for i in range(num_bins):
        bin_number = i + 1
        BE, ell_avg, var_squared= compute_vals_cupy(blcoord_bins[i], vis_bins[i], sigma_0, V_0, bin_number) 

        bare_estimators[i] = BE
        average_ells[i] = ell_avg
        square_variances[i] = var_squared
    end_time = time.time()
    #print('Time to run the bare estimator: ', end_time-start_time, 's')
    
    return bare_estimators, average_ells, square_variances

***
## __All functions for the multiprocessing option.__

In [3]:
def compute_vals(bin_values, sigma_0, V_0, bin_number):
    bin_len = len(bin_values)

    if bin_len == 0 or bin_len == 1:
        return 0, 0, 0

    be_numerator = 0
    be_denominator = 0
    ell_numerator = 0
    ell_denominator = 0
    cntr = 0
    Eb_square_mean_numerator = 0
    Eb_mean_square_numerator = 0
    Eb_denominator = 0

    pair_combs = list(itertools.combinations(range(bin_len), 2))
    for i in range(len(pair_combs)):
        distance = bin_values[pair_combs[i][0]][0] - bin_values[pair_combs[i][1]][0]
        distance_norm = np.linalg.norm(distance)

        if distance_norm <= sigma_0:
            continue

        cntr += 1
        weight = np.exp(-(distance_norm**2) / sigma_0**2)
        visibility_product = bin_values[pair_combs[i][0]][2] * np.conj(bin_values[pair_combs[i][1]][2])
        be_numerator += weight * visibility_product
        be_denominator += weight * V_0 * np.exp(-(distance_norm**2) / (sigma_0**2))

        u_i = bin_values[pair_combs[i][0]][0][0]
        v_i = bin_values[pair_combs[i][0]][0][1]
        l_i = 2 * np.pi * np.sqrt(u_i**2 + v_i**2)
        ell_numerator += weight * np.exp(-(distance_norm**2) / (sigma_0**2)) * l_i
        ell_denominator += weight * np.exp(-(distance_norm**2) / (sigma_0**2))

        Eb_square_mean_numerator += (weight * np.exp(-(distance_norm**2) / (sigma_0**2)) * 5.13 * 1e-4 * (1000 / l_i) ** 2.34) ** 2
        Eb_mean_square_numerator += (weight * np.exp(-(distance_norm**2) / (sigma_0**2)) * 5.13 * 1e-4 * (1000 / l_i) ** 2.34)
        Eb_denominator += weight * np.exp(-(distance_norm**2) / (sigma_0**2))

    if distance_norm <= sigma_0:
        return 0, 0, 0
    
    #print("Bin", bin_number, ":Theoretical combinations: ", len(pair_combs),
    #     " - When filtered: ", cntr)

    bare_estimator = np.abs(be_numerator) / be_denominator
    ell_mean = ell_numerator / ell_denominator
    var_squared = (Eb_square_mean_numerator / Eb_denominator - (Eb_mean_square_numerator / Eb_denominator) ** 2)

    return bare_estimator, ell_mean, np.abs(var_squared)


def _splice_values_function_fit(blmag_sorted_transformed, blcoord_sorted, blmag_sorted, vis_sorted, num_bins):
    bins = np.linspace(np.min(blmag_sorted_transformed), np.max(blmag_sorted_transformed), num_bins)
    bin_indices = np.digitize(blmag_sorted_transformed, bins)

    # Initialize a dictionary to store the baseline coordinates, baseline magnitudes and visibility values
    bin_values = {i: [] for i in range(1, num_bins + 1)}
    # Group the values based on the bin indices
    for i, bin_index in enumerate(bin_indices):
        bin_values[bin_index].append((blcoord_sorted[i], blmag_sorted[i], vis_sorted[i]))

    return bin_values


def _splice_values_equal_bins(blcoord_sorted, blmag_sorted, vis_sorted, num_bins):
    # Calculate number of elements per bin
    elements_per_bin = len(blmag_sorted) // num_bins
    remainder = len(blmag_sorted) % num_bins

    bin_values = {}

    # Distribute elements into bins
    start_idx = 0
    for i in range(num_bins):
        end_idx = start_idx + elements_per_bin
        if i < remainder:
            end_idx += 1  # Distribute remainder elements
        bin_values[i + 1] = list(zip(blcoord_sorted[start_idx:end_idx], blmag_sorted[start_idx:end_idx], vis_sorted[start_idx:end_idx],))
        start_idx = end_idx
    return bin_values


def bare_estimator_func_v4(baselines, visibilities, theta_fwhm, num_bins, bin_type):
    theta_0 = 0.6 * theta_fwhm
    V_0 = np.pi * theta_0**2 / 2
    sigma_0 = 0.76 / theta_fwhm
    
    start_time = time.time()
    #Calculating baseline magnitudes and sorting the three arrays according to them
    bl_mag = np.linalg.norm(baselines, axis=1)
    blcoord_blmag_vis = zip(baselines, bl_mag, visibilities)
    sorted_blcoord_blmag_vis = sorted(blcoord_blmag_vis, key=lambda x: x[1])
    blcoord_sorted, blmag_sorted, vis_sorted = zip(*sorted_blcoord_blmag_vis)
    
    print("Length of input arrays: ", len(blmag_sorted))

    if bin_type == "log":
        blmag_sorted_transformed = np.log10(blmag_sorted)
        bin_values = _splice_values_function_fit(blmag_sorted_transformed, blcoord_sorted, blmag_sorted, vis_sorted, num_bins)
        
    elif bin_type == "linear":
        blmag_sorted_transformed = blmag_sorted
        bin_values = _splice_values_function_fit(blmag_sorted_transformed, blcoord_sorted, blmag_sorted, vis_sorted, num_bins)

    elif bin_type == "equal_length":
        bin_values = _splice_values_equal_bins(blcoord_sorted, blmag_sorted, vis_sorted, num_bins)
    else:
        raise ValueError("Unsupported bin_type")
        
    end_time = time.time()
    print('Time to sort and bin values: ', end_time - start_time, 's')
    
    #Verify the bins
    total_elements = 0 # Initialize the total length variable
    for i in range(len(bin_values)):
        bin_len = len(bin_values[i+1])
        total_elements += bin_len
        print(f"Bin {i+1}: {bin_len} elements - Combinations: {int(bin_len*(bin_len-1) / 2)}")
    # Print to verify the results
    print("Total elements after binning: ", total_elements)
    
    start_time = time.time()
    # create a pool of processes
    pool = mp.Pool(processes=num_bins)
    # map the function to the input data and calculate the results
    results = pool.starmap(
        compute_vals, [(bin_values[key], sigma_0, V_0, key) for key in bin_values.keys()]
    )
    pool.close()
    pool.join()
    end_time = time.time()
    print('Time to run the bare estimator: ', end_time-start_time, 's')
    
    # Combine the results from the processes
    bare_estimator = []
    ell_mean = []
    var_squared = []
    for result in results:
        bare_estimator.append(result[0])
        ell_mean.append(result[1])
        var_squared.append(result[2])

    np_bare_estimator = np.array(bare_estimator)
    np_ell_mean = np.array(ell_mean)
    np_var_squared = np.array(var_squared)
    
    # remove zero values from the arrays
    #np_bare_estimator = np_bare_estimator[np_bare_estimator != 0]
    #np_ell_mean = np_ell_mean[np_ell_mean != 0]
    #np_var_squared = np_var_squared[np_var_squared != 0]

    return np_bare_estimator, np_ell_mean, np_var_squared

***
## __All functions for the non-multiprocessing option.__

In [4]:
def bare_estimator_func_v4_noMP(baselines, visibilities, theta_fwhm, num_bins, bin_type):
    theta_0 = 0.6 * theta_fwhm
    V_0 = np.pi * theta_0**2 / 2
    sigma_0 = 0.76 / theta_fwhm
    
    start_time = time.time()
    #Calculating baseline magnitudes and sorting the three arrays according to them
    bl_mag = np.linalg.norm(baselines, axis=1)
    blcoord_blmag_vis = zip(baselines, bl_mag, visibilities)
    sorted_blcoord_blmag_vis = sorted(blcoord_blmag_vis, key=lambda x: x[1])
    blcoord_sorted, blmag_sorted, vis_sorted = zip(*sorted_blcoord_blmag_vis)
    
    print("Length of input arrays: ", len(blmag_sorted))

    if bin_type == "log":
        blmag_sorted_transformed = np.log10(blmag_sorted)
        bin_values = _splice_values_function_fit(blmag_sorted_transformed, blcoord_sorted, blmag_sorted, vis_sorted, num_bins)
        
    elif bin_type == "linear":
        blmag_sorted_transformed = blmag_sorted
        bin_values = _splice_values_function_fit(blmag_sorted_transformed, blcoord_sorted, blmag_sorted, vis_sorted, num_bins)

    elif bin_type == "equal_length":
        bin_values = _splice_values_equal_bins(blcoord_sorted, blmag_sorted, vis_sorted, num_bins)
    else:
        raise ValueError("Unsupported bin_type")
        
    end_time = time.time()
    print('Time to sort and bin values: ', end_time - start_time, 's')
    
    #Verify the bins
    total_elements = 0 # Initialize the total length variable
    for i in range(len(bin_values)):
        bin_len = len(bin_values[i+1])
        total_elements += bin_len
        print(f"Bin {i+1}: {bin_len} elements - Combinations: {int(bin_len*(bin_len-1) / 2)}")
    # Print to verify the results
    print("Total elements after binning: ", total_elements)
    
    start_time = time.time()
    #Running the bare_estimator for every key (bin)
    results= []
    for key in bin_values.keys():
        result= compute_vals(bin_values[key], sigma_0, V_0, key)
        results.append(result)
    end_time = time.time()
    print('Time to run the bare estimator: ', end_time-start_time, 's')
    
    # Combine the results from the processes
    bare_estimator = []
    ell_mean = []
    var_squared = []
    for result in results:
        bare_estimator.append(result[0])
        ell_mean.append(result[1])
        var_squared.append(result[2])

    np_bare_estimator = np.array(bare_estimator)
    np_ell_mean = np.array(ell_mean)
    np_var_squared = np.array(var_squared)
    
    # remove zero values from the arrays
    #np_bare_estimator = np_bare_estimator[np_bare_estimator != 0]
    #np_ell_mean = np_ell_mean[np_ell_mean != 0]
    #np_var_squared = np_var_squared[np_var_squared != 0]

    return np_bare_estimator, np_ell_mean, np_var_squared

In [28]:
if __name__ == "__main__":
    config = read_config("input/config_LOFAR_multistep_multifreq.yaml")
    
    #Setting the timestep list using the parse_timesteps function
    timesteps = parse_timesteps(config["timesteps"])
    # Setting the freqstep list using the parse_timesteps function 
    freqsteps = parse_timesteps(config["freqsteps"]) #parse_timesteps also works for freqsteps
    
    all_visibilities = np.load('LOFAR_material/LOFAR_visibilities.npy', mmap_mode='r')
    all_baselines = np.load('LOFAR_material/LOFAR_baselines.npy', mmap_mode='r')
    all_frequencies = np.load('LOFAR_material/LOFAR_frequencies.npy')
    
    all_lambdas = cst.c.value / all_frequencies
    
    num_times= all_visibilities.shape[0]
    #theta_maj = config["theta_maj"]
    #theta_min = config["theta_min"]
    #theta_fwhm = np.mean([theta_maj, theta_min])    
    filter_baselines_enabled = config.get("filter_baselines_enabled", True)
    
    num_bins = config["num_bins"]
    all_bare_ests  = np.empty((len(freqsteps), len(timesteps), num_bins))
    all_ell_means = np.empty((len(freqsteps), len(timesteps), num_bins))
    all_var_squares = np.empty((len(freqsteps), len(timesteps), num_bins))
    
    uv_range = config["uv_range"]
    
    # Calculate the indices for quarter, half, and three-quarters
    length = len(timesteps)
    quarter_index = length // 4
    half_index = length // 2
    three_quarters_index = 3 * length // 4
    
    start_time = time.time()
    for f_count, freqstep in enumerate(freqsteps):
        print('*' * 50)
        print('Freqstep: ', freqstep, ' - Frequency: ',  all_frequencies[freqstep])
        
        start_time_freq = time.time()
        for t_count, timestep in enumerate(timesteps):
            # Check and print statements at quarter, half, and three-quarters of the loop
            if t_count == quarter_index:
                print(f"Reached a quarter of the array: index {t_count}")
            elif t_count == half_index:
                print(f"Reached half of the array: index {t_count}")
            elif t_count == three_quarters_index:
                print(f"Reached three quarters of the array: index {t_count}")
            
            bls = all_baselines[timestep]
            vis = all_visibilities[timestep, :, f_count]
                        
            # Decides whether to filter the baselines based on the uv range
            if filter_baselines_enabled:
                bls_filter, vis_filter = filter_baselines_cupy(bls, vis, uv_range, uv_range)
            else:
                bls_filter = bls
                vis_filter = vis
            
            # Filters baselines with NaN visibility
            non_nan_indices = np.where(~np.isnan(vis_filter))[0]
            bls_filter = bls_filter[non_nan_indices]
            vis_filter = np.squeeze(vis_filter[non_nan_indices])
            
            # Filters baselines with zero visibility
            non_zero_indices = np.where(vis_filter != 0)[0]
            bls_filter = bls_filter[non_zero_indices]
            vis_filter = np.squeeze(vis_filter[non_zero_indices])
            
            print('Baselines: ', len(bls), ' - After NaN filtering: ', len(bls_filter))
            print('Timestep: ', timesteps)
            # Initializing the setup and the bare estimator computation
            bare_estimator, ell_mean, var_squared = setup(config, bls_filter, vis_filter, all_lambdas[freqstep])
            
            # Squeeze the arrays to remove single-dimensional entries
            all_bare_ests[f_count, t_count] = np.squeeze(bare_estimator)
            all_ell_means[f_count, t_count] = np.squeeze(ell_mean)
            all_var_squares[f_count, t_count] = np.squeeze(var_squared)
        
        end_time_freq = time.time()
        print("Time for one frequency = ", round(end_time_freq - start_time_freq, 3))
        
    end_time = time.time()
    print("Total time to run the analysis = ", round(end_time - start_time, 3))
            
    bare_estimator_avg = np.mean(all_bare_ests, axis=1)
    ell_mean_avg = np.mean(all_ell_means, axis=1)
    var_squared_avg = np.mean(all_var_squares, axis=1)
                
    # Generate filename based on filtering setting
    
    if filter_baselines_enabled:
        filename_plot = (
            f"PLT_LOFAR_BE_{timesteps[0]}to{timesteps[-1]}timesteps_"
            f"{len(freqsteps)}freqsteps_"
            f"uvRngPm{uv_range[-1]}_"
            f"bins{config['num_bins']}_"
            f"{config['binning_method']}.png")
        
        filename_npz= (
            f"NPARR_LOFAR_BE_{timesteps[0]}to{timesteps[-1]}timesteps_"
            f"{len(freqsteps)}freqsteps_"
            f"uvRngPm{uv_range[-1]}_"
            f"bins{config['num_bins']}_"
            f"{config['binning_method']}")
    else:
        filename_plot = (
            f"PLT_LOFAR_BE_{timesteps[0]}to{timesteps[-1]}timesteps_"
            f"{len(freqsteps)}freqsteps_"
            f"nofilter_"
            f"bins{config['num_bins']}_"
            f"{config['binning_method']}.png")
        
        filename_npz =  (
            f"NPARR_LOFAR_BE_{timesteps[0]}to{timesteps[-1]}timesteps_"
            f"{len(freqsteps)}freqsteps_"
            f"nofilter_"
            f"bins{config['num_bins']}_"
            f"{config['binning_method']}")
    
    
    # Plot the results
    #full_file_path_plot = os.path.join(os.path.dirname("output/"), filename_plot)
    #plot(bare_estimator_avg, ell_mean_avg, var_squared_avg, file_path=full_file_path_plot, img_dpi=config['img_dpi'])
    
    """
    # Save the arrays as a numpy file
    full_file_path_npz = os.path.join(os.path.dirname("output/"), filename_npz)
    npz_filename = os.path.splitext(full_file_path_npz)[0] + ".npz"
    np.savez(npz_filename, bare_estimator_avg=bare_estimator_avg, ell_mean_avg=ell_mean_avg, var_squared_avg=var_squared_avg)
    """

**************************************************
Freqstep:  0  - Frequency:  165184020.99609375
Reached a quarter of the array: index 0
Baselines:  1891  - After NaN filtering:  765
Timestep:  [87]
Time for one frequency =  3.806
Total time to run the analysis =  3.807


In [29]:
print(all_bare_ests)

[[[           inf 2.88672363e+04 6.31962000e+06 6.38860360e+07
   5.86805720e+07 9.08508960e+07 3.85359840e+07 3.40729600e+07
   1.60774970e+07 2.41140860e+07]]]


In [30]:
all_visibilities = np.load('LOFAR_material/LOFAR_visibilities.npy', mmap_mode='r')
all_baselines = np.load('LOFAR_material/LOFAR_baselines.npy', mmap_mode='r')

data = np.load("output/ARR_ALL_LOFAR_BE_0to3594timesteps_20freqsteps_nofilter_bins10_equal_length.npz")

all_bare_ests = data["bare_estimator_all"]
all_ell_means = data["ell_mean_all"]
all_var_squares = data["var_squared_all"]
data.close

print(all_bare_ests.shape)
print(all_ell_means.shape)
print(all_var_squares.shape)

bare_estimator_avg = np.mean(all_bare_ests, axis=1)
ell_mean_avg = np.mean(all_ell_means, axis=1)
var_squared_avg = np.mean(all_var_squares, axis=1)

print(bare_estimator_avg.shape)
print(ell_mean_avg.shape)
print(var_squared_avg.shape)

(20, 3595, 10)
(20, 3595, 10)
(20, 3595, 10)
(20, 10)
(20, 10)
(20, 10)


In [31]:
#print(filename_npz)
#filename_npz = 'ARR_ALL_LOFAR_BE_0to3594timesteps_20freqsteps_nofilter_bins10_equal_length'
#full_file_path_npz = os.path.join(os.path.dirname("output/"), filename_npz)
#npz_filename = os.path.splitext(full_file_path_npz)[0] + ".npz"
#np.savez(npz_filename, bare_estimator_all=all_bare_ests, ell_mean_all=all_ell_means, var_squared_all=all_var_squares)

In [32]:
for i in range(20):
    print(50*'-')
    print(i)
    print(bare_estimator_avg[i]), print

--------------------------------------------------
0
[           inf 6.43204963e+13 2.10223934e+11 9.71572651e+07
 8.07024746e+07 1.36514390e+08 2.21682623e+08 2.14590676e+08
 1.36648237e+08 2.97008605e+08]
--------------------------------------------------
1
[           inf 1.89429583e+14 1.06514807e+08 1.78941168e+08
 7.85606102e+07 1.22186466e+08 2.09625176e+08 2.20313899e+08
 1.40784122e+08 2.87260650e+08]
--------------------------------------------------
2
[           inf 1.52822759e+13 1.49597226e+10 1.16154292e+08
 7.84462282e+07 1.09406146e+08 1.94723906e+08 2.22983848e+08
 1.35561066e+08 2.69296039e+08]
--------------------------------------------------
3
[           inf 2.26840327e+13 3.65683916e+12 5.62689610e+07
 9.92126766e+07 1.11132068e+08 1.96772498e+08 2.15422679e+08
 1.35817725e+08 3.08275398e+08]
--------------------------------------------------
4
[           inf 6.00854193e+11 2.25918852e+11 5.00040283e+07
 8.05513049e+07 9.14355030e+07 1.80009708e+08 2.17020829e+

In [33]:
print(ell_mean_avg)

[[            nan   6407.47866517   9933.19770678  15384.4770717
   27953.67534254  47399.38219043  85767.12510431 151680.14720097
  242449.32625174 368595.57090577]
 [            nan   6104.84983959   9537.63319117  14613.62278922
   26153.64108598  44687.12523878  82378.74353377 147856.8204581
  240286.81788943 368234.14751391]
 [            nan   5927.27001933   8991.03484576  13755.87551857
   24521.56607321  42123.16296723  78530.74263408 143964.3160705
  237356.03129781 367421.55844924]
 [            nan   6010.5068629    9120.36282801  13917.0285269
   24826.45444682  42562.10628749  79461.81036705 144959.85154729
  237952.78948192 367892.35852747]
 [            nan   5888.65573238   8918.11778974  13526.88657478
   23849.56378189  41570.58673559  77470.37904533 142881.38755868
  236227.31064847 366799.10442455]
 [  1548.61465425   5825.86250761   8763.5183701   13288.81023503
   22917.07818014  40785.29499902  75964.76783184 141208.43306893
  234702.07271818 366861.64967837]
 [

In [34]:
print(all_bare_ests[0,87,:])

[           inf 2.46088125e+04 5.08588300e+06 5.11365960e+07
 4.69375800e+07 7.29620480e+07 3.08348460e+07 2.73435760e+07
 1.27834330e+07 1.92905620e+07]


In [35]:
print('all_bare_ests.shape: ', all_bare_ests.shape)
print('all_bare_ests[0,:,0].shape: ', all_bare_ests[0,:,0].shape)
print('')

nan_indices = np.where(np.isnan(all_bare_ests[0,:,0]))[0]
print('nan_indices in all_bare_ests[0,:,0]: ', nan_indices)
zero_indices = np.where(all_bare_ests[0,:,0] == 0)[0]
print('zero_indices in all_bare_ests[0,:,0]: ', zero_indices)
print('')

print('average of all_bare_ests[0,:,0]: ', np.average(all_bare_ests[0,:,0]))
print('indices with inf value in all_bare_ests[0,:,0]: ', np.where(np.isinf(all_bare_ests[0,:,0]))[0])
print('all_bare_ests[0,87,0]:', all_bare_ests[0,87,0], '| all_bare_ests[0,321,0]:', all_bare_ests[0,321,0])
#The inf values have to do with ell_mean values being nan!
print('all_ell_means[0,87,0]:', all_ell_means[0,87,0], '| all_ell_means[0,321,0]:', all_ell_means[0,321,0])

print('')
print('all_visibilities.shape: ', all_visibilities.shape)
print('all_baselines.shape: ', all_baselines.shape)

print('')
print('____Indices with NaN VISIBILITY values: ')
nan_idx = np.where(np.isnan(all_visibilities[87,:,0]))[0]
print('nan_idx: ', nan_idx)
print('nan idx len: ', len(nan_idx))
print(all_visibilities[87,0,0])
print(all_visibilities[87,1847,0])

print('____Indices with zero VISIBILITY values: ')
zero_idx = np.where(all_visibilities[87,:,0] == 0)[0]
print('zero idx: ', zero_idx)
print('zero idx len: ', len(zero_idx))

print()
bls_test = all_baselines[87]
vis_test = all_visibilities[87,:,0]
nonnan_idx = np.where(~np.isnan(vis_test))[0]
filter_bls_test = bls_test[nonnan_idx]
filter_vis_test = vis_test[nonnan_idx]
print('nonnan idx len:', len(nonnan_idx))

print('\n What about zero baselines?')
#zero_bl_idx = np.where(all_visibilities[87,:] == 0)[0]
#print(filter_bls_test)
#print(np.linalg.norm(filter_bls_test, axis=1))

all_bare_ests.shape:  (20, 3595, 10)
all_bare_ests[0,:,0].shape:  (3595,)

nan_indices in all_bare_ests[0,:,0]:  []
zero_indices in all_bare_ests[0,:,0]:  []

average of all_bare_ests[0,:,0]:  inf
indices with inf value in all_bare_ests[0,:,0]:  [ 87 321]
all_bare_ests[0,87,0]: inf | all_bare_ests[0,321,0]: inf
all_ell_means[0,87,0]: nan | all_ell_means[0,321,0]: nan

all_visibilities.shape:  (3595, 1891, 20)
all_baselines.shape:  (3595, 1891, 2)

____Indices with NaN VISIBILITY values: 
nan_idx:  [   0    1    2 ... 1787 1846 1847]
nan idx len:  1126
(nan+nanj)
(nan+nanj)
____Indices with zero VISIBILITY values: 
zero idx:  []
zero idx len:  0

nonnan idx len: 765

 What about zero baselines?


In [36]:
print(filter_bls_test)
print(filter_bls_test.shape)
print(filter_bls_test[0,0])
type(filter_bls_test[0,0])

[[ -410.66459079   768.67470531]
 [  594.320459    -884.71827908]
 [ -313.33077887   662.40989753]
 ...
 [47623.34344287 16647.31896102]
 [32513.13874408 40167.27597903]
 [ 6604.44654085 20549.9205609 ]]
(765, 2)
-410.66459079131147


numpy.float64

In [37]:
"""
def compute_vals_cupy_test(blcoord, vis, sigma_0, V_0, bin_number):

    bin_len = len(blcoord)
    if bin_len == 0 or bin_len == 1:
        return 0, 0, 0

    # Create a meshgrid for vectorized pairwise operations
    idx_i, idx_j = np.triu_indices(bin_len, k=1)
    idx_i = cp.asarray(idx_i)
    idx_j = cp.asarray(idx_j)

    # Calculate pairwise distances
    delta= blcoord[idx_i] - blcoord[idx_j]
    distance_norm = cp.linalg.norm(delta, axis=1)
    
    combination_len_unfiltered= len(distance_norm)
    
    # Filter pairs where distance is less than or equal to sigma_0
    valid_pairs = distance_norm > sigma_0
    if not cp.any(valid_pairs):
        return 0, 0, 0

    distance_norm = distance_norm[valid_pairs]
    idx_i = idx_i[valid_pairs]
    idx_j = idx_j[valid_pairs]
    
    combination_len_filtered= len(distance_norm)
    #print("Bin", bin_number, ":Theoretical combinations: ", combination_len_unfiltered,
    #     " - When filtered: ", combination_len_filtered)
    
    # Calculate weights
    weight = cp.exp(-(distance_norm**2) / (sigma_0**2)).astype(cp.float64)

    # Calculate the visibility product
    visibility_product = vis[idx_i] * cp.conj(vis[idx_j])

    #Calclating the bare estimator
    be_numerator = cp.sum(weight * visibility_product).astype(cp.float64)
    be_denominator = cp.sum(weight * V_0 * weight).astype(cp.float64)
    
    non_zero_indices = np.where(weight != 0.)[0]
    print('Nonzero weights', weight[non_zero_indices], ' | Length: ', len(weight[non_zero_indices]))
    print('be numerator: ', be_numerator)
    print('be denominator: ', be_denominator)
    
    #Calculating the mean of ell
    l_i = 2 * cp.pi * cp.sqrt(blcoord[idx_i, 0]**2 + blcoord[idx_i, 1]**2)
    exp_weight_distance = weight * cp.exp(-(distance_norm**2) / (sigma_0**2))

    ell_numerator = cp.sum(exp_weight_distance * l_i).astype(cp.float64)
    ell_denominator = cp.sum(exp_weight_distance).astype(cp.float64)

    #Computing arrays needed for the error (variance)
    Eb_square_mean_numerator = cp.sum((exp_weight_distance * 5.13 * 1e-4 * (1000 / l_i) ** 2.34) ** 2).astype(cp.float64)
    Eb_mean_square_numerator = cp.sum(exp_weight_distance * 5.13 * 1e-4 * (1000 / l_i) ** 2.34).astype(cp.float64)
    Eb_denominator = cp.sum(exp_weight_distance).astype(cp.float64)

    bare_estimator = cp.abs(be_numerator) / be_denominator
    print(bare_estimator)
    ell_mean = ell_numerator / ell_denominator
    var_squared = (Eb_square_mean_numerator / Eb_denominator - (Eb_mean_square_numerator / Eb_denominator) ** 2)

    # Convert results back to numpy arrays
    bare_estimator = cp.asnumpy(bare_estimator)
    ell_mean = cp.asnumpy(ell_mean)
    var_squared = cp.asnumpy(var_squared)

    return bare_estimator, ell_mean, np.abs(var_squared)
"""


def compute_vals_cupy_test(blcoord, vis, sigma_0, V_0, bin_number):

    bin_len = len(blcoord)
    if bin_len == 0 or bin_len == 1:
        return 0, 0, 0

    # Create a meshgrid for vectorized pairwise operations
    idx_i, idx_j = np.triu_indices(bin_len, k=1)
    idx_i = cp.asarray(idx_i)
    idx_j = cp.asarray(idx_j)

    # Calculate pairwise distances
    delta= blcoord[idx_i] - blcoord[idx_j]
    distance_norm = cp.linalg.norm(delta, axis=1)
    
    combination_len_unfiltered= len(distance_norm)
    
    # Filter pairs where distance is less than or equal to sigma_0
    valid_pairs = distance_norm > sigma_0
    if not cp.any(valid_pairs):
        return 0, 0, 0

    distance_norm = distance_norm[valid_pairs]
    idx_i = idx_i[valid_pairs]
    idx_j = idx_j[valid_pairs]
    
    combination_len_filtered= len(distance_norm)
    #print("Bin", bin_number, ":Theoretical combinations: ", combination_len_unfiltered,
    #     " - When filtered: ", combination_len_filtered)
    
    # Calculate weights
    weight = cp.exp(-(distance_norm**2) / (sigma_0**2))

    # Calculate the visibility product
    visibility_product = vis[idx_i] * cp.conj(vis[idx_j])

    #Calclating the bare estimator
    be_numerator = cp.sum(weight * visibility_product)
    be_denominator = cp.sum(weight * V_0 * weight)
    
    non_zero_indices = np.where(weight != 0.)[0]
    print('Nonzero weights', weight[non_zero_indices], ' | Length: ', len(weight[non_zero_indices]))
    print('be numerator: ', be_numerator)
    print('be denominator: ', be_denominator)
    
    #Calculating the mean of ell
    l_i = 2 * cp.pi * cp.sqrt(blcoord[idx_i, 0]**2 + blcoord[idx_i, 1]**2)
    exp_weight_distance = weight * cp.exp(-(distance_norm**2) / (sigma_0**2))

    ell_numerator = cp.sum(exp_weight_distance * l_i)
    ell_denominator = cp.sum(exp_weight_distance)

    #Computing arrays needed for the error (variance)
    Eb_square_mean_numerator = cp.sum((exp_weight_distance * 5.13 * 1e-4 * (1000 / l_i) ** 2.34) ** 2)
    Eb_mean_square_numerator = cp.sum(exp_weight_distance * 5.13 * 1e-4 * (1000 / l_i) ** 2.34)
    Eb_denominator = cp.sum(exp_weight_distance)

    bare_estimator = cp.abs(be_numerator) / be_denominator
    print(bare_estimator)
    ell_mean = ell_numerator / ell_denominator
    var_squared = (Eb_square_mean_numerator / Eb_denominator - (Eb_mean_square_numerator / Eb_denominator) ** 2)

    # Convert results back to numpy arrays
    bare_estimator = cp.asnumpy(bare_estimator)
    ell_mean = cp.asnumpy(ell_mean)
    var_squared = cp.asnumpy(var_squared)

    return bare_estimator, ell_mean, np.abs(var_squared)

def bare_estimator_func_cupy_test(baselines, visibilities, theta_fwhm, num_bins, bin_type, D, lam, precision= 'single'):
    
    # Set data types based on the precision parameter
    if precision == 'single':
        float_dtype = cp.float32
        complex_dtype = cp.complex64
    elif precision == 'double':
        float_dtype = cp.float64
        complex_dtype = cp.complex128
    else:
        raise ValueError("Unsupported precision type. Use 'single' or 'double'.")
    
    #theta_0 = 0.6 * theta_fwhm
    #V_0 = np.pi * theta_0**2 / 2
    #sigma_0 = 0.76 / theta_fwhm
    theta_fwhm = 1.03 * lam / D
    sigma_0 = 0.76 / theta_fwhm
    theta_0 = 0.6 * theta_fwhm
    V_0 = np.pi * theta_0**2 / 2


    start_time = time.time()
    # Step 0: turn arrays into cupy arrays
    baselines= cp.asarray(baselines, dtype= float_dtype)
    visibilities= cp.asarray(visibilities, dtype= complex_dtype)

    # Step 1: Calculate the baseline magnitudes
    bl_mag = cp.linalg.norm(baselines, axis=1)

    # Stack baselines, magnitudes, and visibilities into a single array for sorting
    #combined_array = cp.hstack((baselines[:,0, None], baselines[:,1, None], bl_mag[:, None], visibilities[:, None]))
    combined_array = cp.hstack((baselines[:], bl_mag[:, None], visibilities[:, None]))

    # Sort the combined array based on the magnitudes
    sorted_combined_array = combined_array[cp.argsort(combined_array[:,2])]

    # Split the sorted array back into individual components
    blcoord_sorted = sorted_combined_array[:, 0:2]
    blmag_sorted = sorted_combined_array[:, 2]
    vis_sorted = sorted_combined_array[:, -1]
    
    #print("Length of input arrays: ", len(blmag_sorted))

    if bin_type == "log":
        blmag_sorted_transformed = cp.log10(blmag_sorted)
        blcoord_bins, blmag_bins, vis_bins = _splice_values_function_fit_cupy(blmag_sorted_transformed, blcoord_sorted, blmag_sorted, vis_sorted, num_bins)

    elif bin_type == "linear":
        blmag_sorted_transformed = blmag_sorted
        blcoord_bins, blmag_bins, vis_bins = _splice_values_function_fit_cupy(blmag_sorted_transformed, blcoord_sorted, blmag_sorted, vis_sorted, num_bins)

    elif bin_type == "equal_length":
        blcoord_bins, blmag_bins, vis_bins = _splice_values_equal_bins_cupy(blcoord_sorted, blmag_sorted, vis_sorted, num_bins)
    else:
        raise ValueError("Unsupported bin_type")

    end_time = time.time()
    #print('Time to sort and bin values: ', end_time-start_time, 's')

    # Verify the bins
    #for i, bin in enumerate(blcoord_bins):
    #    print(f"Bin {i+1}: {len(bin)} elements - Combinations: {int(len(bin)*(len(bin)-1) / 2)}")
    # Print to verify the results
    #print("Total elements after binning: ", cp.sum(cp.array([len(bin) for bin in blmag_bins])))

    bare_estimators = np.empty(num_bins, dtype= float_dtype)
    average_ells = np.empty(num_bins, dtype= float_dtype)
    square_variances = np.empty(num_bins, dtype= float_dtype)

    start_time = time.time()
    for i in range(num_bins):
        bin_number = i + 1        
        #if bin_number==1:
        #    print(vis_bins[i])
        
        BE, ell_avg, var_squared= compute_vals_cupy_test(blcoord_bins[i], vis_bins[i], sigma_0, V_0, bin_number) 

        bare_estimators[i] = BE
        average_ells[i] = ell_avg
        square_variances[i] = var_squared
    end_time = time.time()
    #print('Time to run the bare estimator: ', end_time-start_time, 's')
    
    return bare_estimators, average_ells, square_variances

In [39]:
config = read_config("input/config_LOFAR_multistep_multifreq.yaml")
D = config['D']
print(D)
print(all_lambdas[0])
theta_fwhm = 1.03 * all_lambdas[0] / D
print(theta_fwhm)

theta_0 = 0.6 * theta_fwhm
V_0 = np.pi * theta_0**2 / 2
sigma_0 = 0.76 / theta_fwhm

bare_est_test, ell_mean_test, var_sq_test = bare_estimator_func_cupy_test(filter_bls_test, filter_vis_test, theta_fwhm, 10, 'equal_length', D , all_lambdas[0], precision='double')
#bare_est_test, ell_mean_test, var_sq_test = compute_vals_cupy(cp.asarray(filter_bls_test), cp.asarray(filter_vis_test), sigma_0, V_0, 1)

print(bare_est_test)
print(ell_mean_test)
print(var_sq_test)

30.75
1.814899868596185
0.06079176795622993
Nonzero weights [8.02566002e-114 3.78474759e-174 1.97982378e-058 1.02858594e-143
 1.96109460e-041 4.09781943e-137 6.50074720e-035 2.34564022e-303
 7.30295694e-308 1.21552753e-094 1.68624422e-084 7.17964827e-132
 2.93141762e-092 3.31928901e-224 3.26479214e-224 7.43094087e-227
 1.68624422e-084 6.97835325e-126 6.38354802e-249 6.50074720e-035
 6.45405536e-035 1.76032153e-137 1.48224127e-022 6.48308522e-035
 2.37532010e-303 2.34564022e-303 6.47112445e-035 6.45405536e-035
 2.62845047e-087 8.61438747e-307 6.45405536e-035 6.50074720e-035
 6.45405536e-035 8.37441270e-321 5.63170283e-123 1.25310241e-189
 6.12238947e-068 1.43412601e-215 1.70277742e-122 3.78602415e-124
 3.07010581e-181 9.50330551e-287 5.54203822e-143 5.53299881e-105
 4.09781943e-137 1.82489818e-251 1.39188053e-299 3.43718545e-126
 1.57426166e-280 1.24103893e-248 6.50074720e-035 6.47166431e-035
 2.34564022e-303 3.08122676e-091 6.48717824e-035 6.47166431e-035
 1.27150823e-182 3.22664603e-2

In [121]:
afni = 6.4545511 * 1e-35
print(afni)
print(afni**2)

6.454551099999999e-35
4.1661229902511204e-69
